# Cómo entreno el ranker ahora (split temporal)

## El problema que tenía

Antes le pasaba al ranker como positivos todos los libros de train. Pero EASE y todas las tablas de features (afinidad por autor, frecuencias, recencias, etc.) también se armaban con esos mismos libros. O sea, cuando el ranker veía un positivo, el `score_ease` de ese libro ya estaba calculado sabiendo que el lector lo había leído. Le estaba preguntando "¿este libro está en tu historial?" con el historial a la vista.

Lo medí y era tremendo: en el entrenamiento, el positivo mediano le ganaba a todos sus negativos según EASE. En test, el relevante mediano quedaba entre el 5% y el 24% del pool. El ranker aprendía "EASE alto = positivo" y enterraba todo lo que no venía de arriba de EASE (justo lo que traen el autor y la popularidad).

## Cómo lo armo ahora

Para cada lector, de más viejo a más nuevo:

```
[ ........ historia ........ ][ objetivo: 20 ][ test: 20 ]
 \_____________ train _______________________/
```

- **Test** son los últimos 20, como siempre. No se toca.
- **Train** es todo lo demás, como siempre.
- Dentro de train, separo los últimos 20 como **objetivo** y lo anterior queda como **historia**.

Entonces hay dos momentos, y en los dos hago lo mismo: con lo de antes, predigo los 20 siguientes.

| | entrada (EASE, features, pool) | lo que tiene que predecir |
|---|---|---|
| entrenar el ranker | historia | objetivo (los 20 del medio) |
| evaluar / entregar | train completo | test (los últimos 20) |

Es el mismo problema corrido 20 libros para atrás. Así el ranker aprende exactamente lo que después le pido.

Solo entran los lectores con 20 o más libros en train (2.711). Los que tienen menos aportan su historia a EASE y a las tablas, pero no tienen positivos.

## Qué ve el ranker por cada lector

**Positivos:** los libros del objetivo que el pool alcanza. Son más o menos 14 de 20 (el 72%). Los que el pool no trae los saco: en test tampoco se puntúan nunca, y además los negativos salen solo del pool, así que si los dejaba el ranker aprendía que un EASE muy bajo es bueno.

**Negativos:** salen del mismo pool que uso para predecir, pero armado con la historia:

- el top-2000 de EASE (con una B entrenada solo con la historia),
- los libros de los autores de sus últimos 100 libros,
- los 300 más leídos.

Le saco lo que ya leyó en la historia y los 20 del objetivo, y de lo que queda elijo al azar 5 negativos por cada positivo.

Cada lector termina siendo un grupo de 90 a 120 filas. No es el pool entero (~2.500), es una muestra para que no explote el entrenamiento. Y son negativos "difíciles": libros que EASE, el autor o la popularidad ponían como plausibles y que el lector igual no leyó.

Positivos y negativos pasan por `feature_engineering_batch` con las tablas de la historia. Es el mismo camino que al predecir, así que ya no hay una feature que signifique una cosa al entrenar y otra al predecir (antes los positivos venían con el LOO de la notebook 2).

## Después de entrenar

Libero todo lo de la historia y vuelvo a armar el mundo con train completo (`armar_mundo(df_train, ...)`). Con eso evalúo contra test, o en modo entrega exporto EASE para la notebook 4.

## Qué dio

| | nDCG@20 en test |
|---|---|
| pool unido, entrenamiento viejo | 0,1297 |
| pool unido, split temporal | **0,1520** |

+0,022 pareado (p ≈ 10⁻¹²). Mejoró en todos los tramos, los lectores con nDCG = 0 bajaron de 25% a 16,5%, y los cold pasaron de casi 0 a 0,125.

## Lo que falta

Hasta acá todo lo decidí mirando test, y eso no está bien: test es para reportar, no para elegir. El próximo paso es reservar un 20% de los lectores con objetivo como **validación**: su objetivo no entra al entrenamiento y es contra eso que comparo cambios (etiquetas, `label_gain`, rango de EASE, hiperparámetros). Test lo miro una vez por versión.

#### 1. Librerías.

In [1]:
%run "./librerias/librerias.ipynb"

#### 2. Constantes.

In [2]:
#a. Modo de ejecución.
# "validacion" = entreno solo con train, puedo medir nDCG.
# "entrega"    = entreno con train+test, uso todo el historial para predecir.
with open("./constantes/modo.txt") as f:
    MODO = f.read().strip()

assert MODO in ("validacion", "entrega"), f"MODO inválido: {MODO!r}"
print(f"MODO: {MODO}")

MODO: entrega


In [3]:
#b. Otras constantes.
%run "./constantes/constantes.ipynb"

#### 3. Funciones.

In [4]:
%run "./funciones/funciones.ipynb"

#### 4. Lecturas.

In [5]:
#a. Train.
df_train = pd.read_csv(path_train_fe, dtype={"id_lector": str, "id_libro": str})

In [6]:
#b. Test.
if MODO == "validacion":
    df_test = pd.read_csv(path_test_crudo, dtype={"id_lector": str, "id_libro": str})

In [7]:
#c. Dataset a predecir.
df_a_predecir = pd.read_csv(path_a_predecir, dtype={"id_lector": str})

In [8]:
#d. Libros y Lectores.
df_libros = pd.read_csv(path_libros_fe, dtype={"id_libro": str})
df_lectores = pd.read_csv(path_lectores_fe, dtype={"id_lector": str})

#### 5. Preparación previa.

In [9]:
#a. Forma.
print(f"Modo: {MODO} | Train: {df_train.shape}")
if MODO == "validacion":
    print(f"Test: {df_test.shape}")

Modo: entrega | Train: (461407, 146)


In [10]:
#b. Defino las features del modelo.
features_base = [
    "anio_edicion",
    "nacimiento",
    #"edad_al_interactuar",            # En train varía con fecha.dt.year, al predecir es
    #"dias_transcurridos_interaccion", # anio_actual - nacimiento (constante por lector).
    #"anios_transcurridos_edicion",    # Idem: al predecir queda determinada por anio_edicion.
    #"antiguedad_libro_hoy",
    'frecuencia_lector',
    'frecuencia_libro',
    'dias_desde_primera_lectura',
    #'lecturas_por_dia_libro',        # No suma nada ni a favor ni en contra.
    'n_lectores_distintos_autor',
    'n_interacciones_lector_autor',   # reemplazada por prop_lector_autor (dan el mismo nCDG).
    'n_interacciones_lector_genero',  # reemplazada por prop_lector_genero (dan el mismo nCDG).
    'n_autores_distintos_lector',     # reemplazada por prop_autores_distintos (dan el mismo nCDG).
    'n_generos_distintos_lector',     # reemplazada por prop_generos_distintos (dan el mismo nCDG).
    #'rating_prom_id_lector',
    'rating_prom_id_lector_autor',
    'rating_prom_id_lector_genero_libro_agrupado',
    # Las tres de calidad de ítem degradaban el ranking con RF sobre RMSE
    # (0.0697 -> 0.0409). Con LGBM Ranker pasa algo parecido:
    # La mejor feature para predecir rating es la peor para rankear,
    # y cambiar el objetivo mitiga pero no elimina el conflicto.
    #'rating_prom_id_libro',
    #'rating_prom_autor',
    #'rating_prom_genero',
    #'prop_lector_autor',
    #'prop_lector_genero',
    #'prop_autores_distintos',
    #'prop_generos_distintos',
    #'log_pop_media_lector',        # No mejora el nCDG ni lo empeora.
    #'dif_log_pop',                 # No mejora el nCDG ni lo empeora.
    'score_ease',                   # score colaborativo de EASE^R
    'dias_desde_ultima_lectura',    # constante por lector: contexto
    #'anio_ultima_lectura',         # idem
    'antiguedad_relativa',          # esta sí varía entre candidatos
    #'popularidad_decaida_libro',   # lecturas del libro pesadas por recencia: Canibaliza otras.
    #'popularidad_decaida_autor',   # idem, agregado por autor: Canibaliza otras.
    #'tendencia_libro',             # qué parte de su popularidad es reciente: Neutra.
    'score_ease_corto',             # score colaborativo de EASE^R pero calculado sobre los 10 libros recientes del lector.
    'score_ease_reciente',          # score colaborativo de EASE^R pero calculado sobre los 30 libros recientes del lector.
    'score_ease_medio',             # score colaborativo de EASE^R pero calculado sobre los 100 libros recientes del lector.
    #'rank_ease',                   # puesto del candidato en el EASE del lector: no depende del largo del historial. No suma nada.
    'score_ease_ult_dos_anios',     # EASE pero considerando lo que se leyó en los últimos 2 años.
    'dias_desde_ultima_autor',      # recencia del autor del candidato para este lector.
    'dias_desde_ultima_genero',     # idem, por género.
    'recencia_rel_autor',
    'recencia_rel_genero',
    #'afin_reciente_autor',         # cuántos de los últimos 30 libros fueron de este autor
    #'afin_reciente_genero',        # idem, por género
    'dias_de_actividad',            # cuántos días abarca su historial
    'velocidad_lector',             # libros por día
    'libros_90d',                   # actividad en los últimos 90 días
    'tendencia_lector',             # ritmo reciente vs. su propio promedio
]
features_dummies = [c for c in df_train.columns if c.startswith((
    "genero_persona_",
    #"genero_libro_agrupado_",
    #"editorial_agrupada_",
    #"pais_agrupado_"
))]
features = features_base + features_dummies

print(f"{len(features)} features:", features)

30 features: ['anio_edicion', 'nacimiento', 'frecuencia_lector', 'frecuencia_libro', 'dias_desde_primera_lectura', 'n_lectores_distintos_autor', 'n_interacciones_lector_autor', 'n_interacciones_lector_genero', 'n_autores_distintos_lector', 'n_generos_distintos_lector', 'rating_prom_id_lector_autor', 'rating_prom_id_lector_genero_libro_agrupado', 'score_ease', 'dias_desde_ultima_lectura', 'antiguedad_relativa', 'score_ease_corto', 'score_ease_reciente', 'score_ease_medio', 'score_ease_ult_dos_anios', 'dias_desde_ultima_autor', 'dias_desde_ultima_genero', 'recencia_rel_autor', 'recencia_rel_genero', 'dias_de_actividad', 'velocidad_lector', 'libros_90d', 'tendencia_lector', 'genero_persona_-', 'genero_persona_hombre', 'genero_persona_mujer']


#### 6. Universo de candidatos y split del entrenamiento.

In [11]:
#a. Universo de candidatos: libros con alguna interacción y con la metadata de las features.
todos_los_libros = armar_universo(df_libros)

Universo de candidatos: 48,062 (descarto 75 sin metadata)


In [12]:
#b. Libros del universo agrupados por autor, para la fuente de autor del pool.
# Depende del catálogo y no de la historia, así que se arma una sola vez.
libros_por_autor = (
    df_libros[df_libros["id_libro"].isin(set(todos_los_libros))]
    .groupby("autor")["id_libro"].apply(set).to_dict()
)

In [13]:
#c. Separo los últimos N_OBJETIVO libros de cada lector como positivos del ranker.
# Si EASE se entrena con los mismos pares que después son positivos, el score_ease del
# positivo ya vio la respuesta.
df_historia, df_objetivo = separar_historia_objetivo(df_train, N_OBJETIVO)
print(f"Historia: {len(df_historia):,} | Objetivo: {len(df_objetivo):,} "
      f"| Lectores con objetivo: {df_objetivo['id_lector'].nunique():,}")

Historia: 383,547 | Objetivo: 77,860 | Lectores con objetivo: 3,893


#### 7. Mundo de entrenamiento (solo la historia).

In [14]:
#a. Tablas de referencia, EASE e insumos del pool armados solo con la historia.
# Retrieval y features leen estos nombres como globales.
(media_global, caract_lector, caract_libros,
 afinidad_lector_autor_test, afinidad_lector_genero_test,
 B_ease, libros_idx_ease, leidos_por_lector, historiales,
 autores_recientes, generos_recientes, autores_ventana, populares) = armar_mundo(
    df_historia, df_libros, df_lectores, todos_los_libros
)

Nulos caract_lector: ninguno
Nulos caract_libros: {'autor': 78227, 'genero_libro_agrupado': 78220}
Duplicados (lector / libro / lector-autor / lector-genero): 0 0 0 0
Lectores: 11285 | Libros: 128743 | Libros sin interacciones en la base: 84772
Items: 23,693 | Lectores: 10,341 | Interacciones: 363,230
Matriz item x item: 2.25 GB (float32), pico estimado en la inversion: ~6.7 GB
Calculando X^T X ...
Invirtiendo ...
Listo. B: (23693, 23693), 2.25 GB


In [15]:
#b. EASE con los últimos dos años de la historia.
B_ease_dos_anios, libros_idx_ease_dos_anios = entrenar_ease_dos_anios(df_historia)

Items: 2,835 | Lectores: 1,061 | Interacciones: 11,976
Matriz item x item: 0.03 GB (float32), pico estimado en la inversion: ~0.1 GB
Calculando X^T X ...
Invirtiendo ...
Listo. B: (2835, 2835), 0.03 GB


#### 8. Dataset de entrenamiento del ranker.

Positivos: los últimos libros de cada lector que el pool alcanza. Negativos: el resto de ese mismo pool. Los dos pasan por feature_engineering_batch con tablas armadas solo con la historia, que es como el ranker los va a ver al predecir.

In [16]:
#a. Parámetros del muestreo y de la relevancia.
N_POR_POSITIVO = 5               # negativos por cada positivo que el pool alcanza
MAX_NEG_POR_LECTOR = 4000        # con 20 positivos no se llega, quedó por las dudas.
CORTE_ALTO, CORTE_MEDIO = 9, 7   # rating -> relevancia 3 / 2 / 1. No se usa, queda por las dudas.

In [17]:
#b. Positivos: el objetivo de cada lector, con relevancia derivada del rating.
#i. Sin metadata del lector o del libro no hay features.
lectores_con_metadata = set(caract_lector["id_lector"])
df_pos = df_objetivo[
    df_objetivo["id_lector"].isin(lectores_con_metadata)
    & df_objetivo["id_libro"].isin(set(todos_los_libros))
].copy()
print(f"Descarto {len(df_objetivo) - len(df_pos)} positivos sin metadata.")

#ii. Relevancia.
df_pos["relevancia"] = np.where(
    df_pos["rating"] >= CORTE_ALTO, 3,
    np.where(df_pos["rating"] >= CORTE_MEDIO, 2, 1)
)

Descarto 139 positivos sin metadata.


In [18]:
#c. Negativos del mismo pool que la predicción, y me quedo con los positivos que ese pool alcanza.
df_negativos, df_pos = muestrear_negativos_pool(
    df_pos, n_por_positivo=N_POR_POSITIVO, max_negativos_por_lector=MAX_NEG_POR_LECTOR
)

  500/3,891
  1,000/3,891
  1,500/3,891
  2,000/3,891
  2,500/3,891
  3,000/3,891
  3,500/3,891
Positivos en el pool: 53,651 de 77,721 (69.0%)
Lectores con algún positivo: 3,886
Negativos: 268,255 | ratio: 5.00 | grupo máx: 120


In [19]:
#d. Features de positivos y negativos por el mismo camino que en predicción.
#i. Junto los pares con su etiqueta.
df_negativos["relevancia"] = 0
pares = pd.concat(
    [df_pos[["id_lector", "id_libro", "relevancia"]], df_negativos],
    ignore_index=True
)

#ii. feature_engineering_batch solo mira id_lector e id_libro y hace merges left contra tablas
# sin duplicados: el orden de filas se mantiene y la etiqueta se pega por posición.
df_ranker = feature_engineering_batch(
    pares, caract_lector, caract_libros,
    afinidad_lector_autor_test, afinidad_lector_genero_test, media_global
)
assert len(df_ranker) == len(pares), "Un merge duplicó filas"
assert (df_ranker["id_libro"].values == pares["id_libro"].values).all(), "Se desordenaron las filas"
df_ranker["relevancia"] = pares["relevancia"].values

In [20]:
#e. Ordeno por lector y compruebo antes de entrenar.
# LightGBM arma los grupos con bloques contiguos: desordenado, mezcla lectores sin avisar.
df_ranker = df_ranker.sort_values("id_lector", kind="mergesort").reset_index(drop=True)

#i. Un NaN acá es un merge fallido, no un dato faltante.
nulos = df_ranker[features].isna().sum()
assert nulos.sum() == 0, f"NaN en features: {nulos[nulos > 0].to_dict()}"

#ii. Orden y grupos.
assert df_ranker["id_lector"].is_monotonic_increasing, "df_ranker no está ordenado por lector"
sin_pos = (df_ranker.groupby("id_lector")["relevancia"].max() == 0).sum()
assert sin_pos == 0, f"{sin_pos} lectores sin positivos"

print(f"Filas: {len(df_ranker):,}")
print(df_ranker["relevancia"].value_counts().sort_index())

Filas: 321,906
relevancia
0    268255
1     14705
2     24626
3     14320
Name: count, dtype: int64


#### 9. Entrenamiento.

In [21]:
#a. Armo X, y, y el vector de grupos.
# La etiqueta es el rating crudo (0 los negativos): el nDCG lo usa como ganancia.
X = df_ranker[features]
rating_pos = df_pos.set_index(["id_lector", "id_libro"])["rating"]
claves = pd.MultiIndex.from_arrays([df_ranker["id_lector"], df_ranker["id_libro"]])
y = rating_pos.reindex(claves).fillna(0).astype(int).values
assert (y > 0).sum() == (df_ranker["relevancia"] > 0).sum(), "No matchearon todos los positivos"
group = df_ranker.groupby("id_lector", sort=False).size().values

assert group.sum() == len(X), "El vector de grupos no cubre todas las filas"
print(f"X: {X.shape} | grupos: {len(group):,} | filas por grupo: "
      f"min {group.min()}, mediana {int(np.median(group))}, max {group.max()}")

X: (321906, 30) | grupos: 3,886 | filas por grupo: min 6, mediana 84, max 120


In [24]:
#b. Entreno el ranker.
# Ganancia lineal: con la exponencial por defecto, un 10 pesaría 1.000 veces más que un 1.
# Árboles chicos y pasos cortos: elegidos en la grilla de la sección 13 (mitad A elige,
# mitad B confirma: +0.0084 ± 0.0033 ponderado por Kaggle). Con 63 hojas sobreajustaba.
ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    eval_at=[20],
    label_gain=list(range(11)),
    n_estimators=800,
    learning_rate=0.03,
    num_leaves=15,
    min_child_samples=100,
    reg_lambda=1.0,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

ranker.fit(X, y, group=group)
print("Entrenado.")

c:\Users\i_link\AppData\Local\anaconda3\envs\pruebas_api\lib\site-packages\lightgbm\sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


Entrenado.


In [25]:
#c. Importancia de las features.
imp = pd.Series(ranker.feature_importances_, index=features).sort_values(ascending=False)
print(imp.head(15))

frecuencia_libro                 1257
n_lectores_distintos_autor       1117
dias_desde_primera_lectura        914
antiguedad_relativa               661
rating_prom_id_lector_autor       525
score_ease                        513
n_interacciones_lector_autor      467
score_ease_ult_dos_anios          438
dias_desde_ultima_autor           427
score_ease_reciente               410
score_ease_medio                  393
n_interacciones_lector_genero     371
score_ease_corto                  360
dias_desde_ultima_lectura         359
recencia_rel_autor                353
dtype: int32


In [26]:
#d. Exporto el modelo a un path propio, para no pisar los anteriores.
joblib.dump(ranker, path_modelo_lgbmranker_temporal)
print(f"Modelo exportado en: {path_modelo_lgbmranker_temporal}")

Modelo exportado en: ./modelos/modelo_lgbmranker_temporal_entrega.pkl


In [27]:
#e. Libero lo de la historia antes de entrenar el EASE completo (el pico ronda los 7 GB).
#del B_ease, B_ease_dos_anios, pares, X, y
del B_ease, B_ease_dos_anios, pares, X, y
gc.collect()

849

#### 10. Mundo completo.

In [28]:
#a. Lo mismo que en la sección 7, ahora sobre todo df_train: es lo que ve la evaluación y la entrega.
(media_global, caract_lector, caract_libros,
 afinidad_lector_autor_test, afinidad_lector_genero_test,
 B_ease, libros_idx_ease, leidos_por_lector, historiales,
 autores_recientes, generos_recientes, autores_ventana, populares) = armar_mundo(
    df_train, df_libros, df_lectores, todos_los_libros
)

Nulos caract_lector: ninguno
Nulos caract_libros: {'autor': 78227, 'genero_libro_agrupado': 78220}
Duplicados (lector / libro / lector-autor / lector-genero): 0 0 0 0
Lectores: 11285 | Libros: 128743 | Libros sin interacciones en la base: 80676
Items: 26,508 | Lectores: 10,464 | Interacciones: 439,778
Matriz item x item: 2.81 GB (float32), pico estimado en la inversion: ~8.4 GB
Calculando X^T X ...
Invirtiendo ...
Listo. B: (26508, 26508), 2.81 GB


In [29]:
#b. EASE de los últimos dos años sobre la base completa (lo usan la evaluación y la notebook 4).
B_ease_dos_anios, libros_idx_ease_dos_anios = entrenar_ease_dos_anios(df_train)

Items: 4,556 | Lectores: 1,637 | Interacciones: 23,321
Matriz item x item: 0.08 GB (float32), pico estimado en la inversion: ~0.2 GB
Calculando X^T X ...
Invirtiendo ...
Listo. B: (4556, 4556), 0.08 GB


In [30]:
#c. En entrega: exporto EASE para la notebook 4 y corto acá.
# Evaluar contra un test que ya está adentro del entrenamiento no mide nada.
if MODO == "entrega":
    np.save(path_ease_B, B_ease)
    joblib.dump(libros_idx_ease, path_ease_idx)
    print(f"EASE exportado: {path_ease_B} ({B_ease.nbytes / 1e9:.2f} GB)")
    raise KeyboardInterrupt("Corte intencional: modo entrega. Seguí con la notebook 4.")

EASE exportado: ./modelos/ease_B_entrega.npy (2.81 GB)


KeyboardInterrupt: Corte intencional: modo entrega. Seguí con la notebook 4.

#### 11. Evaluación sobre test - nDCG@20.

In [ ]:
#a. Ground truth por lector (por diseño, 20 libros cada uno).
gt_por_lector = {lid: pd.Series(g["rating"].values, index=g["id_libro"].values)
                 for lid, g in df_test.groupby("id_lector")}
print(f"Lectores en test: {len(gt_por_lector):,}")

In [ ]:
#b. Muestra de test estratificada por frecuencia, para que se parezca a los lectores que evalúa Kaggle.
# Sin esto optimizo para lectores livianos (mediana 45) y me miden sobre una población más
# pesada (mediana 72).
freq_train = df_train.groupby("id_lector").size()
bins = [-1, 0, 20, 50, 100, 250, np.inf]

#i. Distribución objetivo: la de los lectores a predecir.
freq_objetivo = df_a_predecir["id_lector"].map(freq_train).fillna(0)
pesos_objetivo = pd.cut(freq_objetivo, bins).value_counts(normalize=True)

#ii. Lectores disponibles en test, con su tramo.
disponibles = pd.DataFrame({"id_lector": df_test["id_lector"].drop_duplicates().sort_values()})
disponibles["freq"] = disponibles["id_lector"].map(freq_train).fillna(0)
disponibles["bucket"] = pd.cut(disponibles["freq"], bins)

#iii. Muestreo cada tramo según el peso objetivo (o todo lo que haya, si no alcanza).
n_total = 1000
partes = []
for bucket, peso in pesos_objetivo.items():
    pool = disponibles[disponibles["bucket"] == bucket]
    n_pedido = int(round(peso * n_total))
    partes.append(pool.sample(n=min(n_pedido, len(pool)), random_state=42))

muestra = pd.concat(partes)
lectores_test_muestra = muestra["id_lector"].tolist()

#iv. Comprobación.
print(f"Muestra: {len(lectores_test_muestra)} lectores")
print(pd.DataFrame({
    "objetivo": pesos_objetivo,
    "muestra": muestra["bucket"].value_counts(normalize=True)
}).round(3))
print("\nMediana de frecuencia — muestra:", muestra["freq"].median())
print("Mediana de frecuencia — Kaggle: ", freq_objetivo.median())

In [ ]:
#c. nDCG@20 del modelo nuevo, sobre el pool unido.
warnings.filterwarnings("ignore", message=".*eval_at.*")

ndcg_temporal = evaluar_ndcg(lectores_test_muestra, ranker, features, retrieval_union)
print("\nnDCG@20: {:.4f} ± {:.4f} (SE)".format(
    ndcg_temporal.mean(), ndcg_temporal.std(ddof=1) / np.sqrt(len(ndcg_temporal))))
np.save("./outputs/ndcg_base_v25.npy", ndcg_temporal)

#### 12. Análisis.

In [ ]:
#a. nDCG por tramo de lector.
res = pd.DataFrame({"id_lector": lectores_test_muestra, "ndcg": ndcg_temporal})
res["freq"] = res["id_lector"].map(freq_train).fillna(0)
res["bucket"] = pd.cut(res["freq"], [-1, 0, 20, 50, 100, 250, np.inf],
                       labels=["cold", "1-20", "21-50", "51-100", "101-250", "250+"])

print(res.groupby("bucket", observed=True)["ndcg"].agg(["mean", "median", "count"]))
print("\nLectores con nDCG = 0:", (res["ndcg"] == 0).mean())

In [ ]:
#b. Dónde quedan los relevantes según de qué fuente vinieron.
#i. Submuestra del 20% dentro de cada tramo (~7 minutos).
#lectores_diag = (muestra.groupby("bucket", observed=True, group_keys=False)
#                 .sample(frac=0.2, random_state=42)["id_lector"].tolist())
#df_puestos = posiciones_gt(lectores_diag, ranker, features)
#df_puestos["tramo_lector"] = df_puestos["id_lector"].map(res.set_index("id_lector")["bucket"])

#ii. Por fuente: cuántos relevantes hay, cuántos llegan al top-20 y en qué puesto quedan.
#df_puestos["en_top20"] = df_puestos["puesto"] < 20
#print(df_puestos.pivot_table(index="tramo_lector", columns="fuente", values="en_top20",
#                             aggfunc=["size", "mean"], observed=True).round(3), "\n")
#print(df_puestos.groupby("fuente")["puesto"].median())

In [ ]:
#c. Comparación pareada, separada por qué tan reciente es el lector.
# Kaggle evalúa casi solo lectores activos en 2024: el promedio general puede esconder lo que pasa ahí.
fin_test = pd.to_datetime(df_test["fecha"], format="%d-%m-%Y").groupby(df_test["id_lector"]).max()
anio_fin = pd.Series(lectores_test_muestra).map(fin_test).dt.year.values
grupo = np.where(anio_fin >= 2024, "2024", np.where(anio_fin >= 2022, "2022-2023", "antes"))

def comparar_por_recencia(a, b):
    x, y = np.load(f"./outputs/ndcg_{a}.npy"), np.load(f"./outputs/ndcg_{b}.npy")
    for g in ["2024", "2022-2023", "antes"]:
        m = grupo == g
        dif = y[m] - x[m]
        print(f"{g:10s} n={m.sum():4d}  {x[m].mean():.4f} -> {y[m].mean():.4f}  "
              f"dif {dif.mean():+.4f} ± {dif.std(ddof=1) / np.sqrt(m.sum()):.4f}")

#comparar_por_recencia("primera_lectura", "lecturas_dia")

In [ ]:
#d. Recall del pool por recencia del lector y tipo de libro.
# ¿El pool trae los libros sin historia que leen los activos, o ya se pierden antes del ranker?
frec_train = df_train["id_libro"].value_counts()
filas = []
for lid, g in zip(lectores_test_muestra, grupo):
    pool = set(retrieval_union(lid))
    for b in gt_por_lector[lid].index:
        filas.append((g, b in pool, frec_train.get(b, 0)))

df_rec = pd.DataFrame(filas, columns=["grupo", "en_pool", "freq_train"])
df_rec["tramo_libro"] = pd.cut(df_rec["freq_train"], [-1, 0, 1, 4, 19, np.inf],
                               labels=["0", "1", "2-4", "5-19", "20+"])
print(df_rec.pivot_table(index="grupo", columns="tramo_libro", values="en_pool",
                         aggfunc="mean", observed=True).round(3))

#### Prueba de hiperparámetros.

In [ ]:
#a. Etiqueta de entrenamiento. La 9e borró y, así que la rearmo desde df_pos.
rating_pos = df_pos.set_index(["id_lector", "id_libro"])["rating"]
claves = pd.MultiIndex.from_arrays([df_ranker["id_lector"], df_ranker["id_libro"]])
y_hiper = rating_pos.reindex(claves).fillna(0).astype(int).values
group_hiper = df_ranker.groupby("id_lector", sort=False).size().values
assert (y_hiper > 0).sum() == (df_ranker["relevancia"] > 0).sum(), "No matchearon todos los positivos"

In [ ]:
#b. Grilla. Learning rate más bajo en todas; lo que varía es cuánto puede crecer cada árbol.
GRILLA = [dict(num_leaves=nl, min_child_samples=mc, learning_rate=0.03,
               n_estimators=1200, reg_lambda=1.0)
          for nl in [15, 31, 63] for mc in [20, 50, 100]]
# Cantidades de árboles a evaluar: entreno con el máximo y predigo usando solo los primeros k.
ARBOLES_A_EVALUAR = [300, 500, 800, 1200]
# Peso de cada grupo de recencia entre los lectores de Kaggle (diagnóstico del 11/10).
PESO_KAGGLE = {"2024": 0.79, "2022-2023": 0.11, "antes": 0.10}

In [ ]:
#c. Features del pool de cada lector de test, una sola vez (~15 min).
# Armar features es lo caro de evaluar; con esto, cada modelo solo predice.
cache = {}
for i, lid in enumerate(lectores_test_muestra, 1):
    if i % 100 == 0 or i == 1:
        print(f"Lector {i}/{len(lectores_test_muestra)}")
    fe = feature_engineering_test(lid, retrieval_union(lid))
    cache[lid] = (fe["id_libro"].values, fe[features].astype(np.float64).to_numpy())

In [ ]:
#d. nDCG@20 desde el cache, con la misma cuenta que evaluar_ndcg.
# num_iteration=None usa todos los árboles; con un número, solo los primeros.
def ndcg_desde_cache(modelo, num_iteration=None):
    salida = []
    for lid in lectores_test_muestra:
        libros, M = cache[lid]
        scores = dict(zip(libros, modelo.predict(M, num_iteration=num_iteration)))
        gt = gt_por_lector[lid]
        ids = list(set(gt.index) | set(scores))
        piso = min(scores.values()) - 1
        salida.append(ndcg_score(np.asarray([[gt.get(b, 0) for b in ids]]),
                                 np.asarray([[scores.get(b, piso) for b in ids]]), k=20))
    return np.array(salida)

#i. Con el modelo de la 9b tiene que dar lo mismo que la 11c. Si no, el cache está mal y frena acá.
chequeo = ndcg_desde_cache(ranker)
print(f"Cache: {chequeo.mean():.4f} | 11c: {ndcg_temporal.mean():.4f}")
assert abs(chequeo.mean() - ndcg_temporal.mean()) < 1e-4, "El cache no reproduce la evaluación"

In [ ]:
#e. Un entrenamiento por tamaño de árbol; cada uno se evalúa con distintas cantidades de árboles.
# Guardo cada resultado apenas termina, por si algo se corta.
resultados = {}
for k, params in enumerate(GRILLA, 1):
    modelo = lgb.LGBMRanker(
        objective="lambdarank", metric="ndcg", eval_at=[20], label_gain=list(range(11)),
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbose=-1, **params,
    )
    modelo.fit(df_ranker[features], y_hiper, group=group_hiper)
    for n in ARBOLES_A_EVALUAR:
        nombre = f"hiper_nl{params['num_leaves']}_mc{params['min_child_samples']}_n{n}"
        resultados[nombre] = ndcg_desde_cache(modelo, num_iteration=n)
        np.save(f"./outputs/ndcg_{nombre}.npy", resultados[nombre])
        print(f"{k}/{len(GRILLA)} {nombre}: {resultados[nombre].mean():.4f}")

In [ ]:
#f. Elijo con una mitad de los lectores y confirmo con la otra.
# Si elijo la mejor de 36 con los mismos lectores con que la mido, el número sale inflado.
#i. Mitades alternadas: la muestra viene ordenada por tramo, así las dos quedan estratificadas.
idx = np.arange(len(lectores_test_muestra))
mitad_a, mitad_b = idx % 2 == 0, idx % 2 == 1

#ii. Cada lector pesa según cuánto pesa su grupo entre los lectores de Kaggle.
n_grupo = pd.Series(grupo).value_counts()
w = np.array([PESO_KAGGLE[g] / n_grupo[g] for g in grupo])

def ponderado(d, m):
    return (d[m] * w[m]).sum() / w[m].sum()

def se_ponderado(d, m):
    ww, mu = w[m], ponderado(d, m)
    return np.sqrt((ww ** 2 * (d[m] - mu) ** 2).sum()) / ww.sum()

#iii. Diferencia contra la base (la 11c). A elige; B confirma.
tabla = pd.DataFrame({
    nombre: {"A_dif": ponderado(r - ndcg_temporal, mitad_a),
             "B_dif": ponderado(r - ndcg_temporal, mitad_b),
             "B_se": se_ponderado(r - ndcg_temporal, mitad_b),
             "total": r.mean(),
             "2024": r[grupo == "2024"].mean()}
    for nombre, r in resultados.items()
}).T.sort_values("A_dif", ascending=False)
print(tabla.round(4))

In [ ]:
#g. Veredicto: la mejor según A se queda solo si en B gana por más de dos errores estándar.
ganadora = tabla.index[0]
fila = tabla.loc[ganadora]
print(f"Ganadora según A: {ganadora}")
print(f"En B: {fila['B_dif']:+.4f} ± {fila['B_se']:.4f}")
print("Se queda." if fila["B_dif"] > 2 * fila["B_se"] else "No alcanza: la 25 sigue siendo la final.")